In [ ]:
import os
from mdsetup import MDSetup

In [ ]:
# === Change to working dir and initialize MDSetup ===
# os.chdir removed — run from workflows/03_csh_transfer/

lammps_setup = MDSetup(
    system_setup="config/setup_mechanical_charge_cvff.yaml",
    simulation_default="config/defaults.yaml",
    simulation_ensemble="config/ensemble.yaml",
    simulation_sampling="config/sampling_mechanical.yaml",
    submission_command="qsub",
)

In [ ]:
CSH_DATASETS = os.listdir('data')
# CSH_DATASETS = ['CaSi-1.2_WSi-1.2_1_IFF.data']

# === 1. Equilibration ===

# check how many runs are already done in cvff foleder
num_run = len(os.listdir('cvff'))+1

for data_file in CSH_DATASETS:
    lammps_setup.prepare_simulation(
        folder_name=f"run{num_run}/{data_file.split('_')[0]}/equilibration",
        ensembles=["em", "npt"],
        simulation_times=[0.1, 1.0],
        initial_systems=[f"data/{data_file}"],
        input_kwargs={},
        copies=0,
        off_set=0,
    )
    lammps_setup.submit_simulation()
    print(f"✅ Submitted equilibration for {data_file}")


In [ ]:
dirs = os.listdir('cvff/run3')
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 1. Density Analysis ===
mean_all = []
std_all = []
ratio_all = []
for dir in dirs:
    try:
        analysis_folder = f"run3/{dir}/equilibration"
        extracted = lammps_setup.analysis_extract_properties(
            analysis_folder=analysis_folder,
            ensemble='01_npt',
            extracted_properties=['density'],
            output_suffix='density',
            time_fraction=0.2,
        )
        avg = extracted.get('01_npt', {}).get("data", {}).get("average", {})
        mean, std = avg.get('density', {}).get("mean"), avg.get('density', {}).get("std")
        mean_all.append(mean)
        std_all.append(std)
        ratio_all.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}/equilibration: {e}")


In [ ]:
dirs = os.listdir('cvff/run2')
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 1. Density Analysis ===
mean_all2 = []
std_all2 = []
ratio_all2 = []
for dir in dirs:
    try:
        analysis_folder = f"run2/{dir}/equilibration"
        extracted = lammps_setup.analysis_extract_properties(
            analysis_folder=analysis_folder,
            ensemble='01_npt',
            extracted_properties=['density'],
            output_suffix='density',
            time_fraction=0.2,
        )
        avg = extracted.get('01_npt', {}).get("data", {}).get("average", {})
        mean, std = avg.get('density', {}).get("mean"), avg.get('density', {}).get("std")
        mean_all2.append(mean)
        std_all2.append(std)
        ratio_all2.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}/equilibration: {e}")


In [ ]:
dirs = os.listdir('cvff/run1/equilibration')
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 1. Density Analysis ===
mean_all3 = []
std_all3 = []
ratio_all3 = []
for dir in dirs:
    try:
        analysis_folder = f"run1/equilibration/{dir}"
        extracted = lammps_setup.analysis_extract_properties(
            analysis_folder=analysis_folder,
            ensemble='01_npt',
            extracted_properties=['density'],
            output_suffix='density',
            time_fraction=0.2,
        )
        avg = extracted.get('01_npt', {}).get("data", {}).get("average", {})
        mean, std = avg.get('density', {}).get("mean"), avg.get('density', {}).get("std")
        mean_all3.append(mean)
        std_all3.append(std)
        ratio_all3.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}: {e}")


In [ ]:
# pcff density
# === Change to working dir and initialize MDSetup ===
# os.chdir removed — run from workflows/03_csh_transfer/

lammps_setup = MDSetup(
    system_setup="config_pcff/setup_mechanical_charge_pcff.yaml",
    simulation_default="config_pcff/defaults.yaml",
    simulation_ensemble="config_pcff/ensemble.yaml",
    simulation_sampling="config_pcff/sampling_mechanical.yaml",
    submission_command="qsub",
)
dirs = os.listdir('pcff/run7')
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 1. Density Analysis ===
mean_all_pcff = []
std_all_pcff = []
ratio_all_pcff = []
for dir in dirs:
    try:
        analysis_folder = f"run7/{dir}/equilibration"
        extracted = lammps_setup.analysis_extract_properties(
            analysis_folder=analysis_folder,
            ensemble='01_npt',
            extracted_properties=['density'],
            output_suffix='density',
            time_fraction=0.2,
        )
        avg = extracted.get('01_npt', {}).get("data", {}).get("average", {})
        mean, std = avg.get('density', {}).get("mean"), avg.get('density', {}).get("std")
        mean_all_pcff.append(mean)
        std_all_pcff.append(std)
        ratio_all_pcff.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}/equilibration: {e}")


In [ ]:
# === plotting ===

import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

fig, ax = plt.subplots(figsize=(4.5,4))

ax.errorbar(
    ratio_all2,
    mean_all2,
    yerr=std_all2,        # <-- standard deviation
    fmt='o',
    markersize=4,
    color='black',
    ecolor='gray',
    elinewidth=1,
    capsize=2,
    label='C-S-H'
)

ax.plot([1.7, 1.8],[2.604, 2.604], '--', label='C-S-H (Exp)')

ax.plot([0.83],[2.22], label='Tob14 (MD)')
ax.plot([0.83],[2.23], label='Tob14 (Exp)')


ax.plot([0.67],[2.38], label='Tob11M (MD)')
ax.plot([0.67],[2.46], label='Tob11M (Exp)')

ax.plot([0.75],[2.40], label='Tob11M (MD)')
ax.plot([0.75],[2.39], label='Tob11M (Exp)')

# Labels
ax.set_xlabel('Ca/Si ratio', fontsize=12)
ax.set_ylabel('Density (g/cm³)', fontsize=12)


plt.legend()
plt.tight_layout()
plt.show()
# plt.savefig('density_vs_ratio_0.png', dpi=300)


In [ ]:
# === plotting ===

import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

c_csh = '#1b4f72'      # deep blue
c_t14 = '#a93226'      # muted red
c_t11m = '#1e8449'     # muted green
c_t11h = '#6c3483'     # muted purple

fig, ax = plt.subplots(figsize=(4.5,4))

# -------- C–S–H (MD - IFF) --------
ax.errorbar(
    ratio_all2,
    mean_all2,
    yerr=std_all2,
    fmt='o',
    markersize=5,
    color=c_csh,          # dark blue
    ecolor=c_csh,
    elinewidth=1,
    capsize=3,
    label='C–S–H (MD)'
)

# -------- C–S–H Experimental Range --------
ax.plot(
    [1.7, 1.8],
    [2.604, 2.604],
    linestyle='--',
    marker='o',
    markerfacecolor='none',
    markeredgecolor='black',
    markersize=8,
    markeredgewidth=2,
    color='black',
    linewidth=1.5,
    label='C–S–H (Exp.)'
)

# -------- 14 Å --------
ax.plot(0.83, 2.22, '^', color=c_t14, markersize=8,
        label='14 Å (MD)')
ax.plot(0.83, 2.23, '^', markerfacecolor='none',
        markeredgecolor=c_t14, markeredgewidth=2, markersize=8,
        label='14 Å (Exp.)')

# -------- 11 Å (Merlino) --------
ax.plot(0.67, 2.38, '^', color=c_t11m, markersize=8,
        label='11 Å (M) (MD)')
ax.plot(0.67, 2.46, '^', markerfacecolor='none',
        markeredgecolor=c_t11m, markeredgewidth=2, markersize=8,
        label='11 Å (M) (Exp.)')

# -------- 11 Å (Hamid) --------
ax.plot(0.75, 2.40, '^', color=c_t11h, markersize=8,
        label='11 Å (H) (MD)')
ax.plot(0.75, 2.39, '^', markerfacecolor='none',
        markeredgecolor=c_t11h, markeredgewidth=2, markersize=8,
        label='11 Å (H) (Exp.)')

# Labels
ax.set_xlabel('Ca/Si ratio')
ax.set_ylabel('Density (g/cm³)')

ax.set_ylim([2.2,2.65])
# ax.set_xlim([0.55,2.1])

handles, labels = ax.get_legend_handles_labels()

# Define the exact order you want
order = [
    labels.index('14 Å (MD)'),
    labels.index('14 Å (Exp.)'),
    labels.index('11 Å (M) (MD)'),
    labels.index('11 Å (M) (Exp.)'),
    labels.index('11 Å (H) (MD)'),
    labels.index('11 Å (H) (Exp.)'),
    labels.index('C–S–H (MD)'),
    labels.index('C–S–H (Exp.)'),
]

ax.legend([handles[i] for i in order],
          [labels[i] for i in order],
          frameon=False,
          fontsize=9,
          loc='upper left',
          bbox_to_anchor=(0.2, 1))


plt.tight_layout()
# plt.show()
plt.savefig('density_vs_ratio.png', dpi=400)

In [ ]:
import numpy as np

def compute_density_lammps(datafile):
    """
    Compute density (g/cm^3) from a LAMMPS .data file.
    Works for triclinic boxes and atom_style full.
    """

    NA = 6.02214076e23  # Avogadro number

    with open(datafile, 'r') as f:
        lines = f.readlines()

    # ---------------------------
    # Extract box dimensions
    # ---------------------------
    xlo = xhi = ylo = yhi = zlo = zhi = None

    for line in lines:
        if "xlo xhi" in line:
            xlo, xhi = map(float, line.split()[:2])
        if "ylo yhi" in line:
            ylo, yhi = map(float, line.split()[:2])
        if "zlo zhi" in line:
            zlo, zhi = map(float, line.split()[:2])

    Lx = xhi - xlo
    Ly = yhi - ylo
    Lz = zhi - zlo

    V_angstrom3 = Lx * Ly * Lz
    V_cm3 = V_angstrom3 * 1e-24

    # ---------------------------
    # Extract masses
    # ---------------------------
    masses = {}
    mass_section = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("Masses"):
            mass_section = True
            continue

        if mass_section:
            if stripped == "":
                continue
            if stripped.startswith("Atoms"):
                break

            parts = stripped.split()
            if parts[0].isdigit():
                masses[int(parts[0])] = float(parts[1])

    # ---------------------------
    # Extract atom types
    # ---------------------------
    atom_types = []
    atoms_section = False

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("Atoms"):
            atoms_section = True
            continue

        if atoms_section:
            if stripped == "":
                continue
            if stripped.startswith("Bonds"):
                break

            parts = stripped.split()
            if parts[0].isdigit():
                atom_types.append(int(parts[2]))  # atom type column

    # ---------------------------
    # Compute total mass
    # ---------------------------
    total_mass_mol = sum(masses[t] for t in atom_types)
    total_mass_g = total_mass_mol / NA

    density = total_mass_g / V_cm3

    return density, V_angstrom3, total_mass_mol

import glob
import os
base_path = "data"

rho_all = []
casi_all = []

for filepath in glob.glob(os.path.join(base_path, "*.data")):
# for dir in 'data':
    rho, volume, mass_mol = compute_density_lammps(filepath)
    rho_all.append(rho)
    casi_value = float(filepath.split("CaSi-")[1].split("_")[0])
    casi_all.append(casi_value)

_casi_all, _rho_all = zip(*sorted(zip(casi_all, rho_all)))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

# ===============================
# Font setup
# ===============================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)
rcParams['font.family'] = fm.FontProperties(fname=arial_path).get_name()

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

# Colors
# c_csh = '#1f4e79'
c_csh = '#1f77b4'
c_t14 = '#a93226'
c_t11m = '#1e8449'
c_t11h = '#6c3483'

# c_csh  = '#1b3b6f'   # dark navy
# c_t14  = '#8c2d1f'   # deep brick
# c_t11m = '#1b5e3c'   # forest green
# c_t11h = '#4b2e83'   # deep violet


fig, ax = plt.subplots(figsize=(4.5,4))

# ===============================
# C–S–H (MD)
# ===============================
# ax.errorbar(
#     ratio_all2,
#     mean_all2,
#     yerr=std_all2,
#     fmt='o',
#     markersize=7,
#     color=c_csh,
#     ecolor=c_csh,
#     elinewidth=1,
#     capsize=3
# )

ax.plot(ratio_all2, mean_all2,
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_csh,
        label='IFF-MD')


# ===============================
# C–S–H (pyCSH)
# ===============================
plt.scatter(
    _casi_all[:-1],
    _rho_all[:-1],
    marker='x',
    s=65,
    facecolors='k',
    edgecolors='k',
    linewidths=1.2,
    label='C–S–H (pyCSH)'
)

# ===============================
# C–S–H (Exp.)
# ===============================
ax.plot([1.7, 1.8],
        [2.604, 2.604],
        linestyle='--',
        marker='o',
        markerfacecolor='none',
        markeredgecolor=c_csh,
        markersize=7,
        markeredgewidth=1.5,
        color=c_csh,
        linewidth=1.2)


# ===============================
# Tobermorite polymorphs
# ===============================
# 14 Å
ax.plot(0.83, 2.22, '^', color=c_t14, markersize=8)
ax.plot(0.83, 2.23, '^', markerfacecolor='none',
        markeredgecolor=c_t14, markeredgewidth=1.8,
        markersize=8)

# 11 Å (Merlino)
ax.plot(0.67, 2.38, '^', color=c_t11m, markersize=8)
ax.plot(0.67, 2.46, '^', markerfacecolor='none',
        markeredgecolor=c_t11m, markeredgewidth=1.8,
        markersize=8)

# 11 Å (Hamid)
ax.plot(0.75, 2.40, '^', color=c_t11h, markersize=8)
ax.plot(0.75, 2.39, '^', markerfacecolor='none',
        markeredgecolor=c_t11h, markeredgewidth=1.8,
        markersize=8)

# ===============================
# Labels
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Density (g/cm³)', fontsize=14)

# ax.set_ylim(2.2, 2.65)
# ax.set_xlim(0.6, 2.05)

# ===============================
# Clean grouped legend
# ===============================

# IFF-MD (filled circle + filled triangle)
iff_md = (
    Line2D([0], [0], marker='^', color='black',
           markerfacecolor='black',
           linestyle='None', markersize=7),
    Line2D([0], [0], marker='o', color='black',
           markerfacecolor='black',
           linestyle='None', markersize=7)
)

# Experimental (open circle + open triangle)
exp_marker = (
    Line2D([0], [0], marker='^', color='black',
           markerfacecolor='none',
           linestyle='None', markersize=7),
    Line2D([0], [0], marker='o', color='black',
           markerfacecolor='none',
           linestyle='None', markersize=7),
)


legend_elements = [

    # Polymorph color encoding
    Line2D([0], [0], marker='^', color=c_t14,
           linestyle='None', markersize=8,
           label='Tobermorite 14 Å'),

    Line2D([0], [0], marker='^', color=c_t11m,
           linestyle='None', markersize=8,
           label='Tobermorite 11 Å (M)'),

    Line2D([0], [0], marker='^', color=c_t11h,
           linestyle='None', markersize=8,
           label='Tobermorite 11 Å (H)'),

    Line2D([0], [0], marker='o', color=c_csh,
           linestyle='None', markersize=7,
           label='C–S–H'),

    Line2D([0], [0], marker='x', color='k',
           linestyle='None', markersize=7,
           label='pyCSH'),

    Line2D([], [], linestyle='none', label=''),

    Line2D([0], [0], marker=None, color='black',
           markerfacecolor='black',
           linestyle='None', markersize=7,
           label='Filled = IFF-MD'),

    Line2D([0], [0], marker=None, color='black',
           markerfacecolor='none',
           linestyle='None', markersize=7,
           label='Open = Exp.')
#     # Fill encoding
#     (iff_md, 'IFF-MD'),
#     (exp_marker, 'Exp.')
#     Line2D([0], [0], marker='o', color='black',
#            markerfacecolor='black',
#            linestyle='None', markersize=7,
#            label='IFF-MD'),

#     Line2D([0], [0], marker='o', color='black',
#            markerfacecolor='none',
#            linestyle='None', markersize=7,
#            label='Exp.')
]


ax.legend(
    handles=[le[0] if isinstance(le, tuple) else le for le in legend_elements],
    labels=[le[1] if isinstance(le, tuple) else le.get_label() for le in legend_elements],
    handler_map={tuple: HandlerTuple(ndivide=None)},
    frameon=True,
    fontsize=10,
    loc='upper left',
    bbox_to_anchor=(0.15, 1)
)

ax.set_ylim(2.2, 2.72)
ax.set_xlim(0.6, 2.15)

# ax.legend(handles=legend_elements,
#           frameon=True,
#           fontsize=10,
#           loc='upper left',
#           bbox_to_anchor=(0.15, 1))

plt.tight_layout()
plt.savefig('density_vs_ratio_clean_new_cvff.png', dpi=400)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

# ===============================
# Font setup
# ===============================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)
rcParams['font.family'] = fm.FontProperties(fname=arial_path).get_name()

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

# Colors
c_csh_pcff = '#0b3c5d'
c_csh_cvff = '#1f77b4'
c_t14 = '#a93226'
c_t11m = '#1e8449'
c_t11h = '#6c3483'

# c_csh_cvff  = '#1b3b6f'   # dark navy
# c_t14  = '#8c2d1f'   # deep brick
# c_t11m = '#1b5e3c'   # forest green
# c_t11h = '#4b2e83'   # deep violet


fig, ax = plt.subplots(figsize=(4.5,4))

# ===============================
# C–S–H (MD)
# ===============================
# ax.errorbar(
#     ratio_all2,
#     mean_all2,
#     yerr=std_all2,
#     fmt='o',
#     markersize=7,
#     color=c_csh_cvff,
#     ecolor=c_csh_cvff,
#     elinewidth=1,
#     capsize=3
# )

ax.plot(ratio_all2, mean_all2,
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_csh_cvff,
        label='IFF-MD')


ax.plot(ratio_all_pcff[:-1], mean_all_pcff[:-1],
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_csh_pcff,
        label='IFF-MD')

# ===============================
# C–S–H (pyCSH)
# ===============================
plt.scatter(
    _casi_all[:-1],
    _rho_all[:-1],
    marker='x',
    s=65,
    facecolors='k',
    edgecolors='k',
    linewidths=1.2,
    label='C–S–H (pyCSH)'
)

# ===============================
# C–S–H (Exp.)
# ===============================
ax.plot([1.7, 1.8],
        [2.604, 2.604],
        linestyle='--',
        marker='o',
        markerfacecolor='none',
        markeredgecolor=c_csh_cvff,
        markersize=7,
        markeredgewidth=1.5,
        color=c_csh_cvff,
        linewidth=1.2)


# ===============================
# Tobermorite polymorphs
# ===============================
# 14 Å
ax.plot(0.83, 2.22, '^', color=c_t14, markersize=8)
ax.plot(0.83, 2.23, '^', markerfacecolor='none',
        markeredgecolor=c_t14, markeredgewidth=1.8,
        markersize=8)

# 11 Å (Merlino)
ax.plot(0.67, 2.38, '^', color=c_t11m, markersize=8)
ax.plot(0.67, 2.46, '^', markerfacecolor='none',
        markeredgecolor=c_t11m, markeredgewidth=1.8,
        markersize=8)

# 11 Å (Hamid)
ax.plot(0.75, 2.40, '^', color=c_t11h, markersize=8)
ax.plot(0.75, 2.39, '^', markerfacecolor='none',
        markeredgecolor=c_t11h, markeredgewidth=1.8,
        markersize=8)

# ===============================
# Labels
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Density (g/cm³)', fontsize=14)

# ax.set_ylim(2.2, 2.65)
# ax.set_xlim(0.6, 2.05)

# ===============================
# Clean grouped legend
# ===============================

# IFF-MD (filled circle + filled triangle)
iff_md = (
    Line2D([0], [0], marker='^', color='black',
           markerfacecolor='black',
           linestyle='None', markersize=7),
    Line2D([0], [0], marker='o', color='black',
           markerfacecolor='black',
           linestyle='None', markersize=7)
)

# Experimental (open circle + open triangle)
exp_marker = (
    Line2D([0], [0], marker='^', color='black',
           markerfacecolor='none',
           linestyle='None', markersize=7),
    Line2D([0], [0], marker='o', color='black',
           markerfacecolor='none',
           linestyle='None', markersize=7),
)


legend_elements = [

    # Polymorph color encoding
    Line2D([0], [0], marker='^', color=c_t14,
           linestyle='None', markersize=8,
           label='Tobermorite 14 Å'),

    Line2D([0], [0], marker='^', color=c_t11m,
           linestyle='None', markersize=8,
           label='Tobermorite 11 Å (M)'),

    Line2D([0], [0], marker='^', color=c_t11h,
           linestyle='None', markersize=8,
           label='Tobermorite 11 Å (H)'),

    Line2D([0], [0], marker='o', color=c_csh_cvff,
           linestyle='None', markersize=7,
           label='C–S–H (CVFF)'),

    Line2D([0], [0], marker='o', color=c_csh_pcff,
           linestyle='None', markersize=7,
           label='C–S–H (PCFF)'),

    Line2D([0], [0], marker='x', color='k',
           linestyle='None', markersize=7,
           label='pyCSH'),

    Line2D([], [], linestyle='none', label=''),

    Line2D([0], [0], marker=None, color='black',
           markerfacecolor='black',
           linestyle='None', markersize=7,
           label='Filled = IFF-MD'),

    Line2D([0], [0], marker=None, color='black',
           markerfacecolor='none',
           linestyle='None', markersize=7,
           label='Open = Exp.')
#     # Fill encoding
#     (iff_md, 'IFF-MD'),
#     (exp_marker, 'Exp.')
#     Line2D([0], [0], marker='o', color='black',
#            markerfacecolor='black',
#            linestyle='None', markersize=7,
#            label='IFF-MD'),

#     Line2D([0], [0], marker='o', color='black',
#            markerfacecolor='none',
#            linestyle='None', markersize=7,
#            label='Exp.')
]


ax.legend(
    handles=[le[0] if isinstance(le, tuple) else le for le in legend_elements],
    labels=[le[1] if isinstance(le, tuple) else le.get_label() for le in legend_elements],
    handler_map={tuple: HandlerTuple(ndivide=None)},
    frameon=True,
    fontsize=10,
    loc='upper left',
    bbox_to_anchor=(0.15, 1)
)

ax.set_ylim(2.2, 2.72)
ax.set_xlim(0.6, 2.15)

# ax.legend(handles=legend_elements,
#           frameon=True,
#           fontsize=10,
#           loc='upper left',
#           bbox_to_anchor=(0.15, 1))

plt.tight_layout()
plt.savefig('density_vs_ratio_clean_new.png', dpi=400)
plt.show()


In [ ]:
# === plotting ===

import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

c_csh = '#1b4f72'      # deep blue
c_t14 = '#a93226'      # muted red
c_t11m = '#1e8449'     # muted green
c_t11h = '#6c3483'     # muted purple

fig, ax = plt.subplots(figsize=(4.5,4))

# -------- C–S–H (MD - IFF) --------
ax.errorbar(
    ratio_all2,
    mean_all2,
    yerr=std_all2,
    fmt='o',
    markersize=5,
    color='#1f4e79',          # dark blue
    ecolor='#1f4e79',
    elinewidth=1,
    capsize=3,
    label='C–S–H (MD)'
)

# -------- C–S–H Experimental Range --------
ax.plot(
    [1.7, 1.8],
    [2.604, 2.604],
    linestyle='--',
    marker='x',
    markersize=7,
    markeredgewidth=2,
    color='black',
    linewidth=1.5,
    label='C–S–H (Exp.)'
)

# -------- 14 Å --------
ax.plot(0.83, 2.22, 'o', color=c_t14, markersize=6,
        label='14 Å (MD)')
ax.plot(0.83, 2.23, 's', markerfacecolor='none',
        markeredgecolor=c_t14, markeredgewidth=2.5, markersize=6,
        label='14 Å (Exp.)')

# -------- 11 Å (Merlino) --------
ax.plot(0.67, 2.38, 'o', color=c_t11m, markersize=6,
        label='11 Å (M) (MD)')
ax.plot(0.67, 2.46, 's', markerfacecolor='none',
        markeredgecolor=c_t11m, markeredgewidth=2.5, markersize=6,
        label='11 Å (M) (Exp.)')

# -------- 11 Å (Hamid) --------
ax.plot(0.75, 2.40, 'o', color=c_t11h, markersize=6,
        label='11 Å (H) (MD)')
ax.plot(0.75, 2.39, 's', markerfacecolor='none',
        markeredgecolor=c_t11h, markeredgewidth=2.5, markersize=6,
        label='11 Å (H) (Exp.)')

# Labels
ax.set_xlabel('Ca/Si ratio')
ax.set_ylabel('Density (g/cm³)')

ax.set_ylim([2.2,2.65])
# ax.set_xlim([0.55,2.1])

handles, labels = ax.get_legend_handles_labels()

# Define the exact order you want
order = [
    labels.index('14 Å (MD)'),
    labels.index('14 Å (Exp.)'),
    labels.index('11 Å (M) (MD)'),
    labels.index('11 Å (M) (Exp.)'),
    labels.index('11 Å (H) (MD)'),
    labels.index('11 Å (H) (Exp.)'),
    labels.index('C–S–H (MD)'),
    labels.index('C–S–H (Exp.)'),
]

ax.legend([handles[i] for i in order],
          [labels[i] for i in order],
          frameon=False,
          fontsize=9,
          loc='upper left',
          bbox_to_anchor=(0.2, 1))


plt.tight_layout()
# plt.show()
plt.savefig('density_vs_ratio.png', dpi=400)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial for Nature-style) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)


# Nature-like figure settings
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

fig, ax = plt.subplots(figsize=(4.5,4))  # Nature figures are often small

ax.plot(ratio_all, mean_all, 'o', markersize=4, color='black')

# Labels
ax.set_xlabel('Ca/Si ratio', fontsize=12)
ax.set_ylabel('Density (g/cm³)', fontsize=12)

# Remove title (Nature rarely uses titles on plots)
# ax.set_title('Density vs Ca/Si ratio')

# Clean up spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ---------- Remove top ticks ----------

ax.tick_params(top=False)
ax.tick_params(right=False)

plt.tight_layout()

plt.show()


In [ ]:
# === Change to working dir and initialize MDSetup ===
# os.chdir removed — run from workflows/03_csh_transfer/

lammps_setup = MDSetup(
    system_setup="config/setup_mechanical_charge_cvff_deform.yaml",
    simulation_default="config/defaults.yaml",
    simulation_ensemble="config/ensemble.yaml",
    simulation_sampling="config/sampling_mechanical.yaml",
    submission_command="qsub",
)


In [ ]:
# === 2. Deformation ===

CSH_DATASETS = os.listdir('data')

deformation_directions = ["xx", "yy", "zz", "xy", "xz", "yz", "undeformed"]
deformation_rates = [-0.02, -0.01, 0.00, 0.01, 0.02]
num_run = len(os.listdir('cvff'))
for data_file in CSH_DATASETS:
    # if 'CaSi-2.1' in data_file:
    #     print('skipping')
    # else:
    initial_systems = [f"cvff/run{num_run}/{data_file.split('_')[0]}/equilibration/temp_298.1_pres_1.0/copy_0/01_npt/npt.data"]
    job_files = [[] for _ in lammps_setup.system_setup["temperature"]]

    for direction in deformation_directions:
        for rate in deformation_rates:
            if (rate == 0.0 and direction != "undeformed") or (rate != 0.0 and direction == "undeformed"):
                continue
            else:
                folder = (f"run{num_run}/{data_file.split('_')[0]}/deformation/{direction}/{rate}")
                input_kwargs = {"deformation": {"direction": direction, "rate": rate}}

                lammps_setup.prepare_simulation(
                    folder_name=folder,
                    ensembles=["nvt"],
                    simulation_times=[1.0],
                    initial_systems=initial_systems,
                    input_kwargs=input_kwargs,
                    copies=0,
                    off_set=0,
                )
                for j, files in enumerate(lammps_setup.job_files):
                    print(j)
                    job_files[j].extend(files)

    lammps_setup.job_files = job_files
    lammps_setup.submit_simulation(individual_sub=False)
    print(f"✅ Deformation jobs for {data_file}")


In [ ]:
dirs = os.listdir('cvff/run2')
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 1. BM Analysis ===
bm_mean_all = []
bm_std_all = []
bm_ratio_all = []
for dir in dirs:
    try:
        analysis_folder = f"run2/{dir}/deformation"
        BM, BM_std, _, _ = lammps_setup.analysis_mechanical_proerties(
            analysis_folder=analysis_folder,
            ensemble="00_nvt",
            deformation_rates=[-0.02, -0.01, 0.00, 0.01, 0.02],
            method="VRH",
            time_fraction=0.4,
            visualize_stress_strain=False,
        )

        bm_mean_all.append(BM)
        bm_std_all.append(BM_std)
        bm_ratio_all.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}/equilibration: {e}")


In [ ]:
dirs = os.listdir('cvff/run3')
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 1. BM Analysis ===
bm_mean_all2 = []
bm_std_all2 = []
bm_ratio_all2 = []
for dir in dirs:
    try:
        analysis_folder = f"run3/{dir}/deformation"
        BM, BM_std, _, _ = lammps_setup.analysis_mechanical_proerties(
            analysis_folder=analysis_folder,
            ensemble="00_nvt",
            deformation_rates=[-0.02, -0.01, 0.00, 0.01, 0.02],
            method="VRH",
            time_fraction=0.4,
            visualize_stress_strain=False,
        )

        bm_mean_all2.append(BM)
        bm_std_all2.append(BM_std)
        bm_ratio_all2.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}/equilibration: {e}")


In [ ]:
# 2. BM PCFF Analysis
# === Change to working dir and initialize MDSetup ===
# os.chdir removed — run from workflows/03_csh_transfer/

lammps_setup = MDSetup(
    system_setup="config_pcff/setup_mechanical_charge_pcff_deform.yaml",
    simulation_default="config_pcff/defaults.yaml",
    simulation_ensemble="config_pcff/ensemble.yaml",
    simulation_sampling="config_pcff/sampling_mechanical.yaml",
    submission_command="qsub",
)

dirs = os.listdir('pcff/run7')
dirs = [d for d in dirs if d != 'check']
dirs.sort(key=lambda x: (x.split('_')[0].split('-')[1]))

# === 2. BM Analysis ===
bm_mean_all_pcff = []
bm_std_all_pcff = []
bm_ratio_all_pcff = []
for dir in dirs:
    try:
        analysis_folder = f"run7/{dir}/deformation"
        BM, BM_std, _, _ = lammps_setup.analysis_mechanical_proerties(
            analysis_folder=analysis_folder,
            ensemble="00_nvt",
            deformation_rates=[-0.02, -0.01, 0.00, 0.01, 0.02],
            method="VRH",
            time_fraction=0.4,
            visualize_stress_strain=False,
        )

        bm_mean_all_pcff.append(BM)
        bm_std_all_pcff.append(BM_std)
        bm_ratio_all_pcff.append(float(dir.split('_')[0].split('-')[1]))
        print(f"✅ Analysis done for {analysis_folder}: {mean} ± {std}")
    except Exception as e:
        print(f"❌ Analysis failed for {dir}/equilibration: {e}")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

fig, ax = plt.subplots(figsize=(4.5,4))  # Nature figures are often small
ax.plot(bm_ratio_all, bm_mean_all, 'o', markersize=4, color='black', label='data sample 1')
# ax.plot(bm_ratio_all2, bm_mean_all2, 'o', markersize=4, color='red', label='data sample 2')

# plot std bar ar well for each mean point 'bm_std_all' store the STD data
# Labels
ax.set_xlabel('Ca/Si ratio', fontsize=12)
ax.set_ylabel('BM (GPa)', fontsize=12)

# Clean up spines
# ax.spines['top'].set_visible(False)
# ax.spines['right'].set_visible(False)

# ---------- Remove top ticks ----------
# ax.tick_params(top=False)
# ax.tick_params(right=False)

plt.tight_layout()
# plt.legend()
plt.savefig('BM_vs_ratio.png', dpi=400)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===== Font setup (Arial) =====
arial_path = "arial.ttf"
arial_prop = fm.FontProperties(fname=arial_path)
rcParams['font.family'] = arial_prop.get_name()
fm.fontManager.addfont(arial_path)
plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

fig, ax = plt.subplots(figsize=(4.5,4))


c_csh = '#1b4f72'      # deep blue
c_t14 = '#a93226'      # muted red
c_t11m = '#1e8449'     # muted green
c_t11h = '#6c3483'     # muted purple


# ===============================
# 1️⃣ IFF MD present work
# ===============================
ax.plot(bm_ratio_all, bm_mean_all, 'o', color=c_csh, linestyle='-', linewidth=2, label='C–S–H (MD)')

# -------- 14 Å (Tobermorite) --------
ax.plot(0.83, 50.1, '^', color=c_t14, markersize=8, label='14 Å (MD)')
ax.plot(0.83, 47.0, '^',
        markerfacecolor='none',
        markeredgecolor=c_t14,
        markeredgewidth=2,
        markersize=8,
        label='14 Å (Exp.)')

# -------- 11 Å (Merlino) --------
ax.plot(0.67, 79.9, '^', color=c_t11m, markersize=8, label='11 Å (M) (MD)')
ax.plot(0.67, 71, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11m,
        markeredgewidth=2,
        markersize=8,
        label='11 Å (M) (Exp.)')

# -------- 11 Å (Hamid) --------
ax.plot(0.75, 57.9, '^', color=c_t11h, markersize=8, label='11 Å (H) (MD)')
ax.plot(0.75, 55.3, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11h,
        markeredgewidth=2,
        markersize=8,
        label='11 Å (H) (Exp.)')

# ===============================
# 2️⃣ Aslam et al. (AKM)
# ===============================
ax.scatter([1.20, 1.40],
           [73.08, 73.98],
           color='red', marker='o',
           s=80, label='Aslam et al.')

ax.scatter([0.83],
           [49.08],
           color='red', marker='^',
           s=100, label='Aslam et al. (Tobermorite)')

# ===============================
# 3️⃣ Geng et al.
# ===============================
ax.scatter([0.80, 1.00, 1.30],
           [58.34, 69.66, 77.21],
           color='navy', marker='o',
           s=80, label='Geng et al.')

# ===============================
# 4️⃣ Qomi simulation
# ===============================
ax.scatter([1.00, 1.30, 1.50, 1.70],
           [59.96, 55.01, 52.04, 45.03],
           color='lightgray', marker='o',
           s=80, label='Qomi (simulation)')

# ===============================
# 5️⃣ Qomi nanoindentation
# ===============================
ax.scatter([1.00, 1.70],
           [61.03, 42.97],
           color='#8c7a2b', marker='o',
           s=80, label='Qomi (nanoindentation)')

# ===============================
# 6️⃣ Oh et al. (Tobermorite)
# ===============================
ax.scatter([0.83],
           [47.01],
           color='teal', marker='^',
           s=100, label='Oh et al. (Tobermorite)')

# ===============================
# Axis formatting
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Bulk modulus (GPa)', fontsize=14)

# ax.set_xlim(0.75, 1.80)
# ax.set_ylim(40, 85)

ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.6, 1))
# from matplotlib.lines import Line2D

# # ===============================
# # Custom legend elements
# # ===============================

# legend_elements = [

#     # ---- Study (color) ----
#     Line2D([0], [0], marker='o', color='w',
#            markerfacecolor='red', markersize=8,
#            label='Present work'),

#     Line2D([0], [0], marker='o', color='w',
#            markerfacecolor='navy', markersize=8,
#            label='Geng et al.'),

#     Line2D([0], [0], marker='o', color='w',
#            markerfacecolor='lightgray', markersize=8,
#            label='Qomi (simulation)'),

#     Line2D([0], [0], marker='o', color='w',
#            markerfacecolor='#8c7a2b', markersize=8,
#            label='Qomi (nanoindentation)'),

#     Line2D([0], [0], marker='o', color='w',
#            markerfacecolor='teal', markersize=8,
#            label='Oh et al.'),

#     # Spacer (empty line)
#     Line2D([], [], linestyle='none', label=''),

#     # ---- Phase (shape) ----
#     Line2D([0], [0], marker='^', color='black',
#            linestyle='None', markersize=8,
#            label='Tobermorite'),

#     Line2D([0], [0], marker='o', color='black',
#            linestyle='None', markersize=8,
#            label='C–S–H'),

# ]

# # ===============================
# # Create legend
# # ===============================

# ax.legend(handles=legend_elements,
#           frameon=True,
#           facecolor='white',
#           edgecolor='black',
#           fontsize=10,
#           loc='upper right')


# plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===============================
# Font setup (Arial)
# ===============================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)
rcParams['font.family'] = fm.FontProperties(fname=arial_path).get_name()

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

# ===============================
# Create figure
# ===============================
fig, ax = plt.subplots(figsize=(4.5,4))

# ===============================
# Color scheme (muted, high-contrast)
# ===============================
c_iff = '#1f77b4'       # present work
c_t14 = '#a93226'
c_t11m = '#1e8449'
c_t11h = '#6c3483'

c_aslam = '#d62728'     # red
c_geng = '#1b1f3b'      # dark navy
c_qomi_sim = '#b0b0b0'  # darker gray
c_qomi_nano = '#8c7a2b' # muted gold


# ===============================
# 1️⃣ IFF-MD (present work) on Tob
# ===============================
# -------- 14 Å (Tobermorite) --------
ax.plot(0.83, 50.1, '^', color=c_t14, markersize=8, label='14 Å')
ax.plot(0.83, 47.0, '^',
        markerfacecolor='none',
        markeredgecolor=c_t14,
        markeredgewidth=2,
        markersize=8)

# -------- 11 Å (Merlino) --------
ax.plot(0.67, 79.9, '^', color=c_t11m, markersize=8, label='11 Å (M)')
ax.plot(0.67, 71, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11m,
        markeredgewidth=2,
        markersize=8)

# -------- 11 Å (Hamid) --------
ax.plot(0.75, 57.9, '^', color=c_t11h, markersize=8, label='11 Å (H)')
ax.plot(0.75, 55.3, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11h,
        markeredgewidth=2,
        markersize=8)

# ===============================
# 1️⃣ IFF-MD (present work)
# ===============================
ax.plot(bm_ratio_all, bm_mean_all,
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_iff,
        label='C-S-H')

# ===============================
# 2️⃣ Aslam et al. (DFT)
# ===============================
ax.scatter([1.20, 1.40],
           [73.08, 73.98],
           facecolors='none',
           edgecolors=c_aslam,
           marker='o',
           s=60,
           linewidths=2,
           label='Aslam (DFT)')

# ax.scatter([1.20, 1.40],
#            [73.08, 73.98],
#            color=c_aslam,
#            marker='o',
#            s=60,
#            alpha=0.9,
#            label='Aslam (DFT)')

# ===============================
# 3️⃣ Geng et al. (XRD)
# ===============================
ax.scatter([0.80, 1.00, 1.30],
           [58.34, 69.66, 77.21],
           facecolors='none',
           edgecolors=c_geng,
           marker='o',
           s=60,
           linewidths=2,
           label='Geng (Exp.)')

# ax.scatter([0.80, 1.00, 1.30],
#            [58.34, 69.66, 77.21],
#            color=c_geng,
#            marker='o',
#            s=60,
#            alpha=0.95,
#            label='Geng (Exp.)')

# ===============================
# 4️⃣ Qomi simulation
# ===============================
# ax.scatter([1.00, 1.30, 1.50, 1.70],
#            [59.96, 55.01, 52.04, 45.03],
#            color=c_qomi_sim,
#            marker='o',
#            s=60,
#            alpha=0.9,
#            label='Qomi (MD)')

# ===============================
# 5️⃣ Qomi nanoindentation
# ===============================
ax.scatter([1.00, 1.70],
           [61.03, 42.97],
           facecolors='none',
           edgecolors=c_qomi_nano,
           marker='o',
           s=60,
           linewidths=2,
           label='Qomi (Exp.)')

# ax.scatter([1.00, 1.70],
#            [61.03, 42.97],
#            color=c_qomi_nano,
#            marker='o',
#            s=60,
#            alpha=0.95,
#            label='Qomi (Exp.)')

# ===============================
# Axis formatting
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Bulk modulus (GPa)', fontsize=14)

# ax.set_xlim(0.75, 2.05)
# ax.set_ylim(35, 80)

# Legend
ax.legend(frameon=True, fontsize=10, loc='upper right')

# plt.tight_layout()
# plt.savefig('bm_vs_ratio.png', dpi=400)
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===============================
# Font setup (Arial)
# ===============================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)
rcParams['font.family'] = fm.FontProperties(fname=arial_path).get_name()

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

# ===============================
# Create figure
# ===============================
fig, ax = plt.subplots(figsize=(4.5,4))

# ===============================
# Color scheme (muted, high-contrast)
# ===============================
c_iff = '#1f77b4'       # present work
c_t14 = '#a93226'
c_t11m = '#1e8449'
c_t11h = '#6c3483'

c_aslam = '#d62728'     # red
c_geng = '#1b1f3b'      # dark navy
c_qomi_sim = '#b0b0b0'  # darker gray
c_qomi_nano = '#8c7a2b' # muted gold


# ===============================
# 1️⃣ IFF-MD (present work) on Tob
# ===============================
# -------- 14 Å (Tobermorite) --------
ax.plot(0.83, 50.1, '^', color=c_t14, markersize=8, label='14 Å')
ax.plot(0.83, 47.0, '^',
        markerfacecolor='none',
        markeredgecolor=c_t14,
        markeredgewidth=2,
        markersize=8)

# -------- 11 Å (Merlino) --------
ax.plot(0.67, 79.9, '^', color=c_t11m, markersize=8, label='11 Å (M)')
ax.plot(0.67, 71, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11m,
        markeredgewidth=2,
        markersize=8)

# -------- 11 Å (Hamid) --------
ax.plot(0.75, 57.9, '^', color=c_t11h, markersize=8, label='11 Å (H)')
ax.plot(0.75, 55.3, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11h,
        markeredgewidth=2,
        markersize=8)

# ===============================
# 1️⃣ IFF-MD (present work)
# ===============================
ax.plot(bm_ratio_all, bm_mean_all,
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_iff,
        label='C-S-H(IFF-MD)')

# ===============================
# 2️⃣ Aslam et al. (DFT)
# ===============================
ax.scatter([1.20, 1.40],
           [73.08, 73.98],
           facecolors='none',
           edgecolors=c_aslam,
           marker='o',
           s=60,
           linewidths=2,
           label='Aslam (DFT)')

# ax.scatter([1.20, 1.40],
#            [73.08, 73.98],
#            color=c_aslam,
#            marker='o',
#            s=60,
#            alpha=0.9,
#            label='Aslam (DFT)')

# ===============================
# 3️⃣ Geng et al. (XRD)
# ===============================
ax.scatter([0.80, 1.00, 1.30],
           [58.34, 69.66, 77.21],
           facecolors='none',
           edgecolors=c_geng,
           marker='o',
           s=60,
           linewidths=2,
           label='Geng (Exp.)')

# ax.scatter([0.80, 1.00, 1.30],
#            [58.34, 69.66, 77.21],
#            color=c_geng,
#            marker='o',
#            s=60,
#            alpha=0.95,
#            label='Geng (Exp.)')

# ===============================
# 4️⃣ Qomi simulation
# ===============================
# ax.scatter([1.00, 1.30, 1.50, 1.70],
#            [59.96, 55.01, 52.04, 45.03],
#            color=c_qomi_sim,
#            marker='o',
#            s=60,
#            alpha=0.9,
#            label='Qomi (MD)')

# ===============================
# 5️⃣ Qomi nanoindentation
# ===============================
ax.scatter([1.00, 1.70],
           [61.03, 42.97],
           facecolors='none',
           edgecolors=c_qomi_nano,
           marker='o',
           s=60,
           linewidths=2,
           label='Qomi (Exp.)')

# ax.scatter([1.00, 1.70],
#            [61.03, 42.97],
#            color=c_qomi_nano,
#            marker='o',
#            s=60,
#            alpha=0.95,
#            label='Qomi (Exp.)')

# ===============================
# Axis formatting
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Bulk modulus (GPa)', fontsize=14)

ax.set_xlim(0.5, 2.15)
ax.set_ylim(29, 85)

# Legend
ax.legend(frameon=True, fontsize=10, loc='upper right')

# plt.tight_layout()
plt.savefig('bm_vs_ratio_cvff.png', dpi=400)
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===============================
# Font setup (Arial)
# ===============================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)
rcParams['font.family'] = fm.FontProperties(fname=arial_path).get_name()

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

# ===============================
# Create figure
# ===============================
fig, ax = plt.subplots(figsize=(4.5,4))

# ===============================
# Color scheme (muted, high-contrast)
# ===============================
# c_iff = '#1f77b4'       # present work
c_csh_pcff = '#0b3c5d'
c_csh_cvff = '#1f77b4'
c_t14 = '#a93226'
c_t11m = '#1e8449'
c_t11h = '#6c3483'

c_aslam = '#d62728'     # red
c_geng = '#1b1f3b'      # dark navy
c_qomi_sim = '#b0b0b0'  # darker gray
c_qomi_nano = '#8c7a2b' # muted gold


# ===============================
# 1️⃣ IFF-MD (present work) on Tob
# ===============================
# -------- 14 Å (Tobermorite) --------
ax.plot(0.83, 50.1, '^', color=c_t14, markersize=8, label='14 Å')
ax.plot(0.83, 47.0, '^',
        markerfacecolor='none',
        markeredgecolor=c_t14,
        markeredgewidth=2,
        markersize=8)

# -------- 11 Å (Merlino) --------
ax.plot(0.67, 79.9, '^', color=c_t11m, markersize=8, label='11 Å (M)')
ax.plot(0.67, 71, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11m,
        markeredgewidth=2,
        markersize=8)

# -------- 11 Å (Hamid) --------
ax.plot(0.75, 57.9, '^', color=c_t11h, markersize=8, label='11 Å (H)')
ax.plot(0.75, 55.3, '^',
        markerfacecolor='none',
        markeredgecolor=c_t11h,
        markeredgewidth=2,
        markersize=8)

# ===============================
# 1️⃣ IFF-MD (present work)
# ===============================
ax.plot(bm_ratio_all, bm_mean_all,
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_csh_cvff,
        label='C-S-H(CVFF)')


ax.plot(bm_ratio_all_pcff[:-1], bm_mean_all_pcff[:-1],
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_csh_pcff,
        label='C-S-H(PCFF)')

# ===============================
# 2️⃣ Aslam et al. (DFT)
# ===============================
ax.scatter([1.20, 1.40],
           [73.08, 73.98],
           facecolors='none',
           edgecolors=c_aslam,
           marker='o',
           s=60,
           linewidths=2,
           label='Aslam (DFT)')

# ax.scatter([1.20, 1.40],
#            [73.08, 73.98],
#            color=c_aslam,
#            marker='o',
#            s=60,
#            alpha=0.9,
#            label='Aslam (DFT)')

# ===============================
# 3️⃣ Geng et al. (XRD)
# ===============================
ax.scatter([0.80, 1.00, 1.30],
           [58.34, 69.66, 77.21],
           facecolors='none',
           edgecolors=c_geng,
           marker='o',
           s=60,
           linewidths=2,
           label='Geng (Exp.)')

# ax.scatter([0.80, 1.00, 1.30],
#            [58.34, 69.66, 77.21],
#            color=c_geng,
#            marker='o',
#            s=60,
#            alpha=0.95,
#            label='Geng (Exp.)')

# ===============================
# 4️⃣ Qomi simulation
# ===============================
# ax.scatter([1.00, 1.30, 1.50, 1.70],
#            [59.96, 55.01, 52.04, 45.03],
#            color=c_qomi_sim,
#            marker='o',
#            s=60,
#            alpha=0.9,
#            label='Qomi (MD)')

# ===============================
# 5️⃣ Qomi nanoindentation
# ===============================
ax.scatter([1.00, 1.70],
           [61.03, 42.97],
           facecolors='none',
           edgecolors=c_qomi_nano,
           marker='o',
           s=60,
           linewidths=2,
           label='Qomi (Exp.)')

# ax.scatter([1.00, 1.70],
#            [61.03, 42.97],
#            color=c_qomi_nano,
#            marker='o',
#            s=60,
#            alpha=0.95,
#            label='Qomi (Exp.)')

# ===============================
# Axis formatting
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Bulk modulus (GPa)', fontsize=14)

ax.set_xlim(0.5, 2.15)
ax.set_ylim(29, 85)

# Legend
ax.legend(frameon=True, fontsize=10, loc='upper right')

# plt.tight_layout()
plt.savefig('bm_vs_ratio.png', dpi=400)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager as fm

# ===============================
# Font setup (Arial)
# ===============================
arial_path = "arial.ttf"
fm.fontManager.addfont(arial_path)
rcParams['font.family'] = fm.FontProperties(fname=arial_path).get_name()

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.top": True,
    "ytick.right": True
})

# ===============================
# Create figure
# ===============================
fig, ax = plt.subplots(figsize=(4.5,4))

# ===============================
# Color scheme (muted, high-contrast)
# ===============================
c_iff = '#1f77b4'       # present work
c_aslam = '#d62728'     # red
c_geng = '#1b1f3b'      # dark navy
c_qomi_sim = '#b0b0b0'  # darker gray
c_qomi_nano = '#8c7a2b' # muted gold

# ===============================
# 1️⃣ IFF-MD (present work)
# ===============================
ax.plot(bm_ratio_all, bm_mean_all,
        marker='o',
        linestyle='-',
        linewidth=1.5,
        markersize=7,
        color=c_iff,
        label='IFF-MD')

# ===============================
# 2️⃣ Aslam et al. (DFT)
# ===============================
ax.scatter([1.20, 1.40],
           [73.08, 73.98],
           color=c_aslam,
           marker='o',
           s=60,
           alpha=0.9,
           label='Aslam (DFT)')

# ===============================
# 3️⃣ Geng et al. (XRD)
# ===============================
ax.scatter([0.80, 1.00, 1.30],
           [58.34, 69.66, 77.21],
           color=c_geng,
           marker='o',
           s=60,
           alpha=0.95,
           label='Geng (Exp.)')

# ===============================
# 4️⃣ Qomi simulation
# ===============================
ax.scatter([1.00, 1.30, 1.50, 1.70],
           [59.96, 55.01, 52.04, 45.03],
           color=c_qomi_sim,
           marker='o',
           s=60,
           alpha=0.9,
           label='Qomi (MD)')

# ===============================
# 5️⃣ Qomi nanoindentation
# ===============================
ax.scatter([1.00, 1.70],
           [61.03, 42.97],
           color=c_qomi_nano,
           marker='o',
           s=60,
           alpha=0.95,
           label='Qomi (Exp.)')

# ===============================
# Axis formatting
# ===============================
ax.set_xlabel('Ca/Si ratio', fontsize=14)
ax.set_ylabel('Bulk modulus (GPa)', fontsize=14)

ax.set_xlim(0.75, 2.05)
ax.set_ylim(35, 80)

# Legend
ax.legend(frameon=True, fontsize=10, loc='upper right')

plt.tight_layout()
# plt.savefig('bm_vs_ratio.png', dpi=400)
plt.show()
